# 01 - Data audit: `loan_approval_dataset.csv`

Audit of the **primary** dataset (public, synthetic, Kaggle). The target is the loan **approval decision** (Approved / Rejected), not default.

The raw file is only read here, never modified. Cleaning lives in `src/loanlens/cleaning.py`; this notebook documents why each step exists.

In [1]:
import pandas as pd

from loanlens import cleaning

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

RAW = cleaning.RAW_PATH
raw = pd.read_csv(RAW)
print(f'File: {RAW.relative_to(cleaning.PROJECT_ROOT)}')
print(f'Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')

File: data\raw\loan_approval_dataset.csv
Shape: 4,269 rows x 13 columns


## 1. Column names and dtypes

`repr` makes stray whitespace visible.

In [2]:
print([c for c in raw.columns])
cols_with_space = [c for c in raw.columns if c != c.strip()]
print(f'\nColumns with leading/trailing whitespace: {len(cols_with_space)} of {raw.shape[1]}')
raw.dtypes.to_frame('dtype')

['loan_id', ' no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' loan_status']

Columns with leading/trailing whitespace: 12 of 13


,dtype
loan_id,int64
no_of_dependents,int64
education,object
self_employed,object
income_annum,int64
loan_amount,int64
loan_term,int64
cibil_score,int64
residential_assets_value,int64
commercial_assets_value,int64


## 2. Whitespace in string values

In [3]:
text_cols = raw.select_dtypes(include='object').columns
ws = pd.DataFrame({
    'distinct_raw_values': {c: sorted(map(repr, raw[c].unique())) for c in text_cols},
    'cells_with_whitespace': {c: int((raw[c] != raw[c].str.strip()).sum()) for c in text_cols},
})
ws_cells = int(ws['cells_with_whitespace'].sum())
ws

,distinct_raw_values,cells_with_whitespace
education,"[' Graduate', ' Not Graduate']",4269
self_employed,"[' No', ' Yes']",4269
loan_status,"[' Approved', ' Rejected']",4269


## 3. Missing values and duplicates

In [4]:
nulls = raw.isna().sum()
print(f'Total missing cells: {int(nulls.sum())}')
dup_full = int(raw.duplicated().sum())
dup_no_id = int(raw.drop(columns=raw.columns[0]).duplicated().sum())
id_unique = raw[raw.columns[0]].is_unique
print(f'Exact duplicate rows: {dup_full}')
print(f'Duplicate rows ignoring loan_id: {dup_no_id}')
print(f'loan_id unique: {id_unique}')

Total missing cells: 0
Exact duplicate rows: 0
Duplicate rows ignoring loan_id: 0
loan_id unique: True


## 4. Value ranges

Working on a whitespace-stripped copy for readability.

In [5]:
df = cleaning.strip_string_values(cleaning.normalise_columns(raw))
num_cols = [c for c in df.select_dtypes('number').columns if c != 'loan_id']
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
no_of_dependents,"4,269.00",2.50,1.70,0.00,1.00,3.00,4.00,5.00
income_annum,"4,269.00","5,059,123.92","2,806,839.83","200,000.00","2,700,000.00","5,100,000.00","7,500,000.00","9,900,000.00"
loan_amount,"4,269.00","15,133,450.46","9,043,362.98","300,000.00","7,700,000.00","14,500,000.00","21,500,000.00","39,500,000.00"
loan_term,"4,269.00",10.90,5.71,2.00,6.00,10.00,16.00,20.00
cibil_score,"4,269.00",599.94,172.43,300.00,453.00,600.00,748.00,900.00
residential_assets_value,"4,269.00","7,472,616.54","6,503,636.59","-100,000.00","2,200,000.00","5,600,000.00","11,300,000.00","29,100,000.00"
commercial_assets_value,"4,269.00","4,973,155.31","4,388,966.09",0.00,"1,300,000.00","3,700,000.00","7,600,000.00","19,400,000.00"
luxury_assets_value,"4,269.00","15,126,305.93","9,103,753.67","300,000.00","7,500,000.00","14,600,000.00","21,700,000.00","39,200,000.00"
bank_asset_value,"4,269.00","4,976,692.43","3,250,185.31",0.00,"2,300,000.00","4,600,000.00","7,100,000.00","14,700,000.00"


In [6]:
for c in ['education', 'self_employed', 'loan_status', 'no_of_dependents', 'loan_term']:
    print(f'{c}:', df[c].value_counts().sort_index().to_dict())
approval_share = (df['loan_status'] == 'Approved').mean()
print(f'\nApproved share: {approval_share:.1%}')

education: {'Graduate': 2144, 'Not Graduate': 2125}
self_employed: {'No': 2119, 'Yes': 2150}
loan_status: {'Approved': 2656, 'Rejected': 1613}
no_of_dependents: {0: 712, 1: 697, 2: 708, 3: 727, 4: 752, 5: 673}
loan_term: {2: 404, 4: 447, 6: 490, 8: 386, 10: 436, 12: 456, 14: 405, 16: 412, 18: 422, 20: 411}

Approved share: 62.2%


## 5. Outliers

IQR rule (outside Q1 - 1.5*IQR, Q3 + 1.5*IQR) per numeric column, plus a check for impossible values (negative amounts, CIBIL outside 300 to 900).

In [7]:
def iqr_outliers(s: pd.Series) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

outliers = pd.DataFrame({
    'iqr_outliers': {c: iqr_outliers(df[c]) for c in num_cols},
    'negative_values': {c: int((df[c] < 0).sum()) for c in num_cols},
    'min': df[num_cols].min(),
    'max': df[num_cols].max(),
})
cibil_out_of_range = int((~df['cibil_score'].between(300, 900)).sum())
print(f'CIBIL outside 300-900: {cibil_out_of_range}')
outliers

CIBIL outside 300-900: 0


,iqr_outliers,negative_values,min,max
no_of_dependents,0,0,0,5
income_annum,0,0,200000,9900000
loan_amount,0,0,300000,39500000
loan_term,0,0,2,20
cibil_score,0,0,300,900
residential_assets_value,52,28,-100000,29100000
commercial_assets_value,37,0,0,19400000
luxury_assets_value,0,0,300000,39200000
bank_asset_value,5,0,0,14700000


In [8]:
lti = df['loan_amount'] / df['income_annum']
print(f'Loan-to-income ratio: min {lti.min():.2f}, max {lti.max():.2f}')

Loan-to-income ratio: min 1.50, max 4.00


IQR flags in skewed money columns are expected for wealthy applicants; they are plausible values, not errors, so they are **kept**. The only impossible values are negative asset values, examined next.

## 6. Negative `residential_assets_value`

In [9]:
neg = df[df['residential_assets_value'] < 0]
n_neg = len(neg)
print(f'Rows with negative residential_assets_value: {n_neg}')
print('Distinct negative values:', neg['residential_assets_value'].value_counts().to_dict())
print('Status split:', neg['loan_status'].value_counts().to_dict())
print(f"Approval rate in these rows: {(neg['loan_status'] == 'Approved').mean():.1%} "
      f"vs overall {approval_share:.1%}")
cols = ['cibil_score', 'income_annum', 'loan_amount', 'residential_assets_value']
neg[cols].describe().T

Rows with negative residential_assets_value: 28
Distinct negative values: {-100000: 28}
Status split: {'Approved': 16, 'Rejected': 12}
Approval rate in these rows: 57.1% vs overall 62.2%


,count,mean,std,min,25%,50%,75%,max
cibil_score,28.00,637.86,193.96,336.00,451.75,643.50,830.75,899.00
income_annum,28.00,"2,767,857.14","2,434,482.77","200,000.00","400,000.00","2,200,000.00","5,025,000.00","7,800,000.00"
loan_amount,28.00,"8,325,000.00","7,849,256.87","300,000.00","1,475,000.00","6,750,000.00","13,875,000.00","29,600,000.00"
residential_assets_value,28.00,"-100,000.00",0.00,"-100,000.00","-100,000.00","-100,000.00","-100,000.00","-100,000.00"


### Comparing treatments

In [10]:
rows = []
for strategy in cleaning.VALID_STRATEGIES:
    out = cleaning.clean(raw, strategy)
    rows.append({
        'strategy': strategy,
        'rows': len(out),
        'approval_rate': out['loan_status'].mean(),
        'residential_min': out['residential_assets_value'].min(),
        'residential_mean': out['residential_assets_value'].mean(),
        'missing_cells': int(out.isna().sum().sum()),
    })
pd.DataFrame(rows)

,strategy,rows,approval_rate,residential_min,residential_mean,missing_cells
0,clip_zero,4269,0.62,0.00,"7,473,272.43",0
1,drop,4241,0.62,0.00,"7,522,612.59",0
2,nan,4269,0.62,0.00,"7,522,612.59",28


**Decision: `clip_zero` (configurable in `config/policy.yaml`, `cleaning.negative_assets_strategy`).**

- Assets cannot be negative, so these are data errors.
- Every negative value is the same number (-100,000). That looks like a sentinel or an entry error, not a real liability.
- Setting them to 0 keeps all rows (they make up under 1% of the data and include both outcomes). It is the conservative reading: no residential assets are credited.
- `drop` changes the approval rate and the residential mean almost not at all, so it is an equally safe alternative. `nan` would push imputation into the model pipeline for under 1% of rows with no benefit.

## 7. Run the cleaning pipeline and save

In [11]:
strategy = cleaning.load_negative_strategy()
clean_df = cleaning.build_clean_dataset()
print(f'Strategy from config: {strategy}')
print(f'Saved {cleaning.PROCESSED_PATH.relative_to(cleaning.PROJECT_ROOT)}: '
      f'{clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns')
assert clean_df.isna().sum().sum() == 0
assert (clean_df[list(cleaning.ASSET_COLUMNS)] >= 0).all().all()
assert 'loan_id' not in clean_df.columns
clean_df.head()

Strategy from config: clip_zero
Saved data\processed\loan_approval_clean.csv: 4,269 rows x 12 columns


,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1
1,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0
2,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0
3,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0
4,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0


## 8. Data-quality summary

In [12]:
summary = pd.DataFrame([
    {'issue': 'Column names with leading/trailing whitespace', 'count': len(cols_with_space),
     'action': 'Stripped and normalised to snake_case (normalise_columns)'},
    {'issue': 'Text cells with leading/trailing whitespace', 'count': ws_cells,
     'action': 'Stripped (strip_string_values)'},
    {'issue': 'Missing values', 'count': int(nulls.sum()), 'action': 'None needed'},
    {'issue': 'Exact duplicate rows', 'count': dup_full,
     'action': 'None found; pipeline still drops exact duplicates'},
    {'issue': 'Duplicate rows ignoring loan_id', 'count': dup_no_id, 'action': 'None needed'},
    {'issue': 'Negative residential_assets_value (all -100,000)', 'count': n_neg,
     'action': f'{strategy}: set to 0 (configurable)'},
    {'issue': 'CIBIL outside 300-900', 'count': cibil_out_of_range, 'action': 'None needed'},
    {'issue': 'IQR outliers (all numeric columns)', 'count': int(outliers['iqr_outliers'].sum()),
     'action': 'Kept: plausible values, not errors'},
    {'issue': 'Target stored as text (Approved/Rejected)', 'count': len(raw),
     'action': 'Mapped to 1/0 (map_target)'},
    {'issue': 'loan_id is an identifier, not a feature', 'count': 1,
     'action': 'Dropped (drop_identifier)'},
])
summary

,issue,count,action
0,Column names with leading/trailing whitespace,12,Stripped and normalised to snake_case (normali...
1,Text cells with leading/trailing whitespace,12807,Stripped (strip_string_values)
2,Missing values,0,None needed
3,Exact duplicate rows,0,None found; pipeline still drops exact duplicates
4,Duplicate rows ignoring loan_id,0,None needed
5,"Negative residential_assets_value (all -100,000)",28,clip_zero: set to 0 (configurable)
6,CIBIL outside 300-900,0,None needed
7,IQR outliers (all numeric columns),94,"Kept: plausible values, not errors"
8,Target stored as text (Approved/Rejected),4269,Mapped to 1/0 (map_target)
9,"loan_id is an identifier, not a feature",1,Dropped (drop_identifier)
